# 📥 Notebook 1 — Data Collection & Structural Cleaning
**Project:** Citizen Grievance Analysis  
**Dataset:** NYC 311 Service Requests  


### What this notebook does:
- Loads the raw CSV(Sample 10K Data) from `data/raw/grievance_sample.csv`
- Inspects shape, columns, null values
- Selects relevant columns
- Fixes missing values
- Parses dates & engineers new features
- Saves cleaned data → `data/cleaned/grievance_cleaned.csv`

---

##  Setup Paths

In [61]:
# Paths for data collection and storage ─────────────────────
RAW_DATA_PATH = '../data/raw/grievance_sample.csv'
OUTPUT_DIR    = '../data/cleaned/'

## Load the Raw CSV

In [62]:
import pandas as pd
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)

print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
df.head(3)

Rows    : 12,387
Columns : 53


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Complaint Type,Descriptor,Location Type,Incident Zip,Incident Address,...,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Garage Lot Name,Ferry Direction,Ferry Terminal Name,Latitude,Longitude,Location
0,32310363,12/31/2015 11:59:45 PM,01/01/2016 12:55:15 AM,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,10034.0,71 VERMILYEA AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.865682,-73.923501,"(40.86568153633767, -73.92350095571744)"
1,32309934,12/31/2015 11:59:44 PM,01/01/2016 01:26:57 AM,NYPD,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,11105.0,27-07 23 AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.775945,-73.915094,"(40.775945312321085, -73.91509393898605)"
2,32309159,12/31/2015 11:59:29 PM,01/01/2016 04:51:03 AM,NYPD,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,10458.0,2897 VALENTINE AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.870325,-73.888525,"(40.870324522111424, -73.88852464418646)"


##  Check Missing Values

In [63]:
null_df = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    'Null %'    : (df.isnull().sum() / len(df) * 100).round(2)
})

# Show only columns that HAVE nulls
print(null_df[null_df['Null Count'] > 0].to_string())

                                Null Count  Null %
Closed Date                           2381   19.22
Descriptor                             189    1.53
Location Type                           66    0.53
Incident Zip                          2335   18.85
Incident Address                      1131    9.13
Street Name                           1131    9.13
Cross Street 1                        3073   24.81
Cross Street 2                        3562   28.76
Intersection Street 1                10794   87.14
Intersection Street 2                11281   91.07
Address Type                          2341   18.90
City                                  2335   18.85
Landmark                             12381   99.95
Facility Type                         2372   19.15
Due Date                                 3    0.02
Resolution Action Updated Date        2400   19.38
X Coordinate (State Plane)            2355   19.01
Y Coordinate (State Plane)            2355   19.01
School or Citywide Complaint   

##  Select Only Useful Columns

In [64]:
# We only need these 10 columns for NLP work
NLP_COLS = [
    'Unique Key',
    'Created Date',
    'Closed Date',
    'Agency Name',
    'Complaint Type',        # ← This is our TARGET LABEL for classification
    'Descriptor',            # ← Main complaint text
    'Location Type',
    'Borough',
    'Status',
    'Resolution Description' # ← How it was resolved
]

df_nlp = df[NLP_COLS].copy()
print(f'Reduced to {df_nlp.shape[1]} columns, {df_nlp.shape[0]:,} rows')
df_nlp.head(3)

Reduced to 10 columns, 12,387 rows


,Unique Key,Created Date,Closed Date,Agency Name,Complaint Type,Descriptor,Location Type,Borough,Status,Resolution Description
0,32310363,12/31/2015 11:59:45 PM,01/01/2016 12:55:15 AM,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,MANHATTAN,Closed,The Police Department responded and upon arriv...
1,32309934,12/31/2015 11:59:44 PM,01/01/2016 01:26:57 AM,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,QUEENS,Closed,The Police Department responded to the complai...
2,32309159,12/31/2015 11:59:29 PM,01/01/2016 04:51:03 AM,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,BRONX,Closed,The Police Department responded and upon arriv...


## Fix Missing Values

In [65]:
TEXT_COLS = [
    'Descriptor', 'Resolution Description',
    'Complaint Type', 'Agency Name', 'Location Type'
]

for col in TEXT_COLS:
    df_nlp[col] = df_nlp[col].fillna('unknown')

# Drop rows where Complaint Type is missing
# (it is our label — we cannot train without it)
df_nlp = df_nlp[df_nlp['Complaint Type'] != 'unknown']

print(f'Rows after fixing nulls: {len(df_nlp):,}')
print('Remaining nulls:')
print(df_nlp.isnull().sum())

Rows after fixing nulls: 12,387
Remaining nulls:
Unique Key                   0
Created Date                 0
Closed Date               2381
Agency Name                  0
Complaint Type               0
Descriptor                   0
Location Type                0
Borough                      0
Status                       0
Resolution Description       0
dtype: int64


## Parse Dates & Engineer Features

In [66]:
import numpy as np

df_nlp['Created Date'] = pd.to_datetime(df_nlp['Created Date'], errors='coerce')
df_nlp['Closed Date']  = pd.to_datetime(df_nlp['Closed Date'],  errors='coerce')

# How many hours to resolve the complaint?  (Keep as float for math/sorting)
df_nlp['resolution_hours'] = (
    (df_nlp['Closed Date'] - df_nlp['Created Date'])
    .dt.total_seconds() / 3600
)

# What hour of the day was complaint filed?
df_nlp['hour_created'] = df_nlp['Created Date'].dt.hour

# What day of the week?
df_nlp['day_of_week'] = df_nlp['Created Date'].dt.day_name()

# --- DATA TRANSFORMATION: RESOLUTION METRICS & FORMATTING ---
# This section standardizes date/time reporting for the final report, 
# explicitly handling 'Assigned' or 'Open' cases as "In Progress".

# 1. Format Closed Date: Convert NaT (Open/Assigned) to 'In Progress' for readability
df_nlp['closed_date_fmt'] = np.where(
    df_nlp['Closed Date'].isna(),
    'In Progress',
    df_nlp['Closed Date'].dt.strftime('%Y-%m-%d %H:%M:%S')
)

# 2. Format Resolution Hours: Categorize based on duration and completion status
# - If NaN: Ticket is still open ("In Progress")
# - If < 1 hour: Label as fast resolution ("<1 hr")
# - Otherwise: Round to 1 decimal place and append unit(like: 3.5hr)
df_nlp['resolution_hours_fmt'] = np.select(
    [
        df_nlp['resolution_hours'].isna(),
        df_nlp['resolution_hours'] < 1 
    ],
    [
        "In Progress",
        "<1 hr"
    ],
    default=df_nlp['resolution_hours'].round(1).astype(str) + " hrs"
)

# 3. Convert creation timestamp to 12-hour AM/PM format
df_nlp['hour_created_fmt'] = df_nlp['Created Date'].dt.strftime('%I %p')

print('New columns added: closed_date_fmt, resolution_hours, resolution_hours_fmt, hour_created, hour_created_fmt, day_of_week')
df_nlp[['Status','Created Date','Closed Date', 'closed_date_fmt', 'resolution_hours', 'resolution_hours_fmt', 'hour_created', 'hour_created_fmt', 'day_of_week']].head()

/tmp/ipykernel_9886/63476920.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_nlp['Created Date'] = pd.to_datetime(df_nlp['Created Date'], errors='coerce')
/tmp/ipykernel_9886/63476920.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_nlp['Closed Date']  = pd.to_datetime(df_nlp['Closed Date'],  errors='coerce')


New columns added: closed_date_fmt, resolution_hours, resolution_hours_fmt, hour_created, hour_created_fmt, day_of_week


,Status,Created Date,Closed Date,closed_date_fmt,resolution_hours,resolution_hours_fmt,hour_created,hour_created_fmt,day_of_week
0,Closed,2015-12-31 23:59:45,2016-01-01 00:55:15,2016-01-01 00:55:15,0.925000,<1 hr,23,11 PM,Thursday
1,Closed,2015-12-31 23:59:44,2016-01-01 01:26:57,2016-01-01 01:26:57,1.453611,1.5 hrs,23,11 PM,Thursday
2,Closed,2015-12-31 23:59:29,2016-01-01 04:51:03,2016-01-01 04:51:03,4.859444,4.9 hrs,23,11 PM,Thursday
3,Closed,2015-12-31 23:57:46,2016-01-01 07:43:13,2016-01-01 07:43:13,7.757500,7.8 hrs,23,11 PM,Thursday
4,Closed,2015-12-31 23:56:58,2016-01-01 03:24:42,2016-01-01 03:24:42,3.462222,3.5 hrs,23,11 PM,Thursday


##  Create Combined Text Column

In [67]:
# Merge complaint type + descriptor + resolution into one text field
# This gives NLP models more signal to work with
df_nlp['combined_text'] = (
    df_nlp['Complaint Type'].astype(str)         + ' ' +
    df_nlp['Descriptor'].astype(str)             + ' ' +
    df_nlp['Resolution Description'].astype(str)
)

print('Sample combined_text:')
print(df_nlp['combined_text'].iloc[0])

Sample combined_text:
Noise - Street/Sidewalk Loud Music/Party The Police Department responded and upon arrival those responsible for the condition were gone.


##  Quick Summary & Save

In [68]:
print('=== FINAL CLEANED DATASET SUMMARY ===')
print(f'Total rows  & columns  : {df_nlp.shape}')
print(f'Unique complaint types (labels): {df_nlp["Complaint Type"].nunique()}')
print(f'Boroughs      : {df_nlp["Borough"].unique().tolist()}')
print(f'Date range    : {df_nlp["Created Date"].min()} → {df_nlp["Created Date"].max()}')

# Save output for Notebook 2
SAVE_PATH = OUTPUT_DIR + 'grievance_cleaned.csv'
df_nlp.to_csv(SAVE_PATH, index=False)
print(f'\n✅ Saved → {SAVE_PATH}')
print('👉 Now open Notebook 2: 02_text_preprocessing.ipynb')

=== FINAL CLEANED DATASET SUMMARY ===
Total rows  & columns  : (12387, 17)
Unique complaint types (labels): 20
Boroughs      : ['MANHATTAN', 'QUEENS', 'BRONX', 'BROOKLYN', 'Unspecified', 'STATEN ISLAND']
Date range    : 2015-01-01 07:30:10 → 2015-12-31 23:59:45

✅ Saved → ../data/cleaned/grievance_cleaned.csv
👉 Now open Notebook 2: 02_text_preprocessing.ipynb
